In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from transliterate import translit
import re
import numpy as np

In [ ]:
def normalize_translit(ru_query):
    text = translit(ru_query, language_code='ru', reversed=True)
    text = text.replace(' ', '-').replace('yj', 'yy').replace('yy-', 'yy_').replace('es-', 'es_').replace('k-pr', 'k_pr')
    return text

In [ ]:
driver = webdriver.Chrome('./chromedriver')

/var/folders/0k/pk7zyw6x2sqcjpbqvgf0ssh80000gn/T/ipykernel_22701/2299280398.py:1: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome('./chromedriver')


In [ ]:
def scrape(query, driver):
    pages_count = 10
    link = 'https://hh.ru/resumes/{}?items_on_page=100&page={}'.format(query, 0)
    driver.get(link)

    pages_cnt_site = driver.find_elements(by=By.XPATH, value='//span[@class="pager-item-not-in-short-range"]/a[@class="bloko-button"][@rel="nofollow"][@data-qa="pager-page"]/span')
    if pages_cnt_site:
        pages_count = int(pages_cnt_site[-1].text)

    for n in range(pages_count):
        link = 'https://hh.ru/resumes/{}?items_on_page=100&page={}'.format(query, n)
        driver.get(link)

        resumes_list = driver.find_element(by=By.XPATH, value='//div[@class="resume-search-item__header"]')
        resumes = resumes_list.find_element(by=By.XPATH, value='//div[@data-qa="resume-serp__results-search"]').find_elements(by=By.XPATH, value='//a[@class="serp-item__title"]')

        resumes_parsed = []
        for x in resumes:
            resumes_parsed.append((x.text, x.get_attribute("href")))

        if n == 0:
            df = pd.DataFrame(resumes_parsed, columns=['Вакансия', 'Ссылка'])
        else:

            df2 = pd.DataFrame(resumes_parsed, columns=['Вакансия', 'Ссылка'])
            df = pd.concat([df, df2])

    return df

In [ ]:
analysts_list = ['аналитик bi',
                'системный аналитик',
                'бизнес аналитик',
                'аналитик продаж',
                'финансовый аналитик',
                'аналитик данных',
                'data analyst']
analysts_map = dict()

for analyst in analysts_list:
    analysts_map[analyst] = scrape(normalize_translit(analyst), driver)
    analysts_map[analyst].index = range(analysts_map[analyst].shape[0])

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//div[@class="resume-search-item__header"]"}
  (Session info: chrome=113.0.5672.92)
Stacktrace:
0   chromedriver                        0x0000000102479670 chromedriver + 4298352
1   chromedriver                        0x0000000102471bbc chromedriver + 4266940
2   chromedriver                        0x00000001020a4758 chromedriver + 280408
3   chromedriver                        0x00000001020dfb38 chromedriver + 523064
4   chromedriver                        0x0000000102118080 chromedriver + 753792
5   chromedriver                        0x00000001020d22d0 chromedriver + 467664
6   chromedriver                        0x00000001020d3354 chromedriver + 471892
7   chromedriver                        0x00000001024396c4 chromedriver + 4036292
8   chromedriver                        0x000000010243dc64 chromedriver + 4054116
9   chromedriver                        0x00000001024442d8 chrome

In [ ]:
driver.get('https://hh.ru/resumes/analitik-bi?items_on_page=100&page=0')

In [ ]:
driver.find_elements(by=By.XPATH,
                     value='//a[@class="bloko-button"][@rel="nofollow"][@data-qa="pager-page"]/span')[-1].text

'25'

In [ ]:
driver.find_elements(by=By.XPATH,
                     value='//div[@class="resume-search-item__content"]//a[@class="serp-item__title"][@data-qa="serp-item__title"][@rel="nofollow"][@target="_blank"]')[-2].text

'Аналитик / Data Scientist'

In [ ]:
driver.find_elements(by=By.XPATH,
                     value='//div[@class="resume-search-item__content"]//a[@class="serp-item__title"][@data-qa="serp-item__title"][@rel="nofollow"][@target="_blank"]')[-2].get_attribute("href")

'https://hh.ru/resume/503dff340008f6c3730039ed1f506131443561?query=%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D1%82%D0%B8%D0%BA+BI&source=search&hhtmFrom=resumes_catalog'

In [ ]:
df1 = pd.DataFrame()

for analyst in analysts_list:
    df2 = analysts_map[analyst]
    df1 = pd.concat([df1, df2])

df1.index = range(df1.shape[0])

In [ ]:
df1

In [ ]:
def normalize(x):
    if x:
        x = x[0].text
    else:
        x = '---'
    return x

In [ ]:
def scrape(link, driver):
    driver.get(link)
    comandirovka = driver.find_elements(by=By.XPATH, value='//div[@class="bloko-translate-guard"]')
    comandirovka = normalize(comandirovka)

    opyt = driver.find_elements(by=By.XPATH, value='//span[@class="resume-block__title-text resume-block__title-text_sub"]')
    opyt = normalize(opyt)

    about = driver.find_elements(by=By.XPATH, value='//div[@class="resume-block-container"][@data-qa="resume-block-skills-content"]')
    about = normalize(about)

    educ = driver.find_elements(by=By.XPATH, value='//div[@class="resume-block"][@data-qa="resume-block-education"]')
    educ = normalize(educ)

    inter = driver.find_elements(by=By.XPATH, value='//div[@class="resume-block-item-gap"]')
    inter = normalize(inter)

    inter2 = driver.find_elements(by=By.XPATH, value='//div[@data-qa="skills-table"]')
    inter2 = normalize(inter2)


    educ_val = '//div[@data-qa="resume-block-education"][@class="resume-block"]//div[@data-qa="resume-block-education-name"]'
    educ_ = driver.find_elements(by=By.XPATH, value=educ_val)
    stepen_val = '//div[@data-qa="resume-block-education"]//span[@class="resume-block__title-text resume-block__title-text_sub"]'
    stepen = driver.find_elements(by=By.XPATH, value=stepen_val)
    title_val = '//span[@class="resume-block__title-text"][@data-qa="resume-block-title-position"]'
    title = driver.find_elements(by=By.XPATH, value=title_val)
    if title:
        title = title[0].text
    else:
        title = ''

    educ1 = ''
    educ2 = ''
    educ3 = ''

    if educ_:
        educ1 = educ_[0].text
    if len(educ_) > 1:
        educ2 = educ_[1].text
    if stepen:
        educ3 = stepen[0].text
    vuzes_count = len(educ_)


    work = driver.find_elements(by=By.XPATH,
                             value='//div[@data-qa="resume-block-experience"][@class="resume-block"]//div[@class="resume-block-container"]')
    work1 = '---'
    work2 = '---'

    if len(work) > 0:
        work1 = work[0].text
    if len(work) > 1:
        work2 = work[1].text

    work_list = []
    for w in work:
        work_list.append(w.text)
    work_count = len(work)

    return (link, title, comandirovka, opyt, about, educ, inter, inter2, educ1, educ2,
            educ3, vuzes_count, work1, work2, work_list, work_count)

In [ ]:
df_links = list(pd.read_csv('resumes_all.csv')['Ссылка'])[:300]

resume_list = []
for link in df_links:
    data = scrape(link, driver)
    resume_list.append(data)

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(resume_list, columns=['Ссылка',
                                        'Название',
                                        'Коммандировка',
                                        'Опыт',
                                        'О себе',
                                        'Образование',
                                        'Интересы',
                                        'Навыки',
                                        'Образование-1',
                                        'Образование-2',
                                        'Уровень образование',
                                        'Кол-во образований',
                                        'Работа 1',
                                        'Работа 2',
                                        'Где работал?',
                                        'Кол-во работ'])
df.head(2)

,Ссылка,Название,Коммандировка,"Опыт работы, лет",О себе,Образование,Интересы,Навыки,Образование-1,Образование-2,Уровень образование,Кол-во образований,Работа 1,Работа 2,Где работал?,Кол-во работ
0,https://hh.ru/resume/1aa91e410000fd3cda0039ed1...,Analyst,"Moscow, willing to relocate, prepared for occa...",Work experience 7 years 4 months,"Responsible, communicable, quick study and det...",Higher education\n2015\nMOSCOW STATE UNIVERSIT...,"Specializations:\nSales manager, account manag...",Key skills\nAnalitical thinking\nEnglish\nOrga...,MOSCOW STATE UNIVERSITY OF MECHANICAL ENGINEER...,,Higher education,1,"NUTRICIA\nMoscow, nutricia.ru\nFood Products.....","Volkswagen Group Russia\nMoscow, www.volkswage...","[NUTRICIA\nMoscow, nutricia.ru\nFood Products....",4
1,https://hh.ru/resume/628596ac000657935b0039ed1...,BI аналитик,"Москва, не готова к переезду, готова к редким ...",Опыт работы 1 год 5 месяцев,В последние годы проходила обучение без возмож...,Высшее образование (Бакалавр)\n2022\nНациональ...,"Специализации:\nBI-аналитик, аналитик данных\n...",Ключевые навыки\nTableau\nPower BI\nSQL\nMS Ex...,"Национальный исследовательский университет ""Вы...",,Высшее образование (Бакалавр),1,"Ozon\nМладший аналитик\nСоздание, поддержка и ...",OZON\nСтажер группы BI аналитики и отчетности\...,"[Ozon\nМладший аналитик\nСоздание, поддержка и...",3
